In [1]:
import numpy as np
import pandas as pd

In [2]:
data_path = "mock_output.csv"

In [3]:
data = pd.read_csv(data_path)

In [4]:
data = data.rename(columns={"used_for_parameter_estimation" : "parameter"})
data = data.rename(columns={"bob_measured_quadrature" : "bob_basis"})

In [5]:
# in the real data we won't already have this so simulate we don't
data = data.drop(["alice_value_for_bob_basis"], axis=1)

### Keep only Alice values corresponding to Bob's measurements

In [6]:
# keep track of when bob measured X and when he measured P
bob_x_idxs = data["bob_basis"] == "X"
bob_p_idxs = data["bob_basis"] == "P"

In [7]:
alice_value_x = data[bob_x_idxs]["alice_x"]
alice_value_p = data[bob_p_idxs]["alice_p"]

In [8]:
alice_value = pd.concat([alice_value_x, alice_value_p])

In [9]:
data.insert(6, "alice_value", alice_value)

In [10]:
data.head()

,pulse_id,alice_x,alice_p,bob_basis,bob_value,parameter,alice_value
0,0,0.609434,-0.903902,X,-2.315105,False,0.609434
1,1,-2.079968,-1.331755,X,-2.134312,False,-2.079968
2,2,1.500902,0.868020,X,0.574552,False,1.500902
3,3,1.881129,0.503709,X,1.668453,False,1.881129
4,4,-3.902070,-2.809583,X,-2.570418,False,-3.902070


### Estimate information rate between Alice and Bob

In [11]:
# only use subset designated for parameter estimation
parameter_subset = data[data["parameter"] == True]

A = parameter_subset["alice_value"]
B = parameter_subset["bob_value"]

### Channel gain

### $g = \frac{\text{Cov}(A,B)}{\text{Var}(A)}$

In [12]:
channel_gain = np.cov(A, B)[0,1] / np.var(A)
print(f"Channel gain g : {channel_gain}")

Channel gain g : 0.8165714135341845


### Noise variance

Variance of the noise that remains after removing the part of Bob's measurement that is explained by Alice's value

#### $\sigma_n^2 = \text{Var}(B - gA)$

In [13]:
noise_var = np.var(B - channel_gain * A)
print(f"The noise variance is : {noise_var}")

The noise variance is : 1.056759005609136


### Signal to noise ratio

### $\Sigma_B = \frac{g^2 \text{Var}(A)}{\sigma_n^2} $

In [14]:
bob_snr = (channel_gain * np.var(A)) / noise_var
print(f"Bob's signal to noise ratio : {bob_snr}")

Bob's signal to noise ratio : 2.9271795225417563


### Information rate between Alice and Bob

### $I_{AB} = \frac{1}{2} \log_2 (1 + \Sigma_B)$

In [15]:
I_AB = 0.5 * (np.log2(1 + bob_snr))
print(f"I_AB = {I_AB}")

I_AB = 0.9867467746019641


### Upper bound on Eve's information

### Input noise

### $\chi = \frac{\text{Var}(n)}{g^2}$

In [16]:
chi = noise_var / (channel_gain**2)
print(f"Chi = {chi}")

Chi = 1.5848479897673238


### Eve signal to noise ratio

### $\Sigma_E = \frac{\text{Var}(A)}{1 + 1/\chi}$

In [17]:
eve_snr = np.var(A) / (1 + (1/chi))
print(f"Eve SNR : {eve_snr}")

Eve SNR : 2.322649882618737


### Information rate between Alice and Eve (upper bound)

Since we don't have Eve's data we cannot directly estimate it, however we can upper bound her SNR based on channel information estimated from the Alice and Bob data that we have, therefore we can upper bound her information rate

### $I_{AE} = \frac{1}{2} \log_2 (1 + \Sigma_E)$

In [18]:
I_AE = 0.5 * (np.log2(1 + eve_snr))
print(f"I_AE = {I_AE}")

I_AE = 0.8661671400116104


### Secret information rate

### $\Delta I = I_{AB} - I_{AE}$

For the protocol we require that $\Delta I > 0$

In [19]:
secret_rate = I_AB - I_AE
print(f"Secret information rate : {secret_rate}")

Secret information rate : 0.1205796345903537


## Converting gaussian variables to bitstrings

### Sliced reconciliation parameters

Each real value will become an m bit bitstring

In [20]:
# number of slices
m = 4
n_intervals = 2**m

In [21]:
A_vals = data["alice_value"]
B_vals = data["bob_value"]

In [22]:
def calculate_thresholds(A_vals, n_intervals):
    """Calculate tresholds for n intervals based on Gaussian quantiles
    Each resulting range has equal probability"""

    probs = np.linspace(0, 1, n_intervals + 1)
    thresholds = np.quantile(A_vals, probs)

    thresholds[0] = -np.inf
    thresholds[-1] = np.inf
    
    return thresholds

In [23]:
thresholds = calculate_thresholds(A_vals, n_intervals)
print(f"Calculated thresholds for {m} slices\n")
print(thresholds)

Calculated thresholds for 4 slices

[       -inf -3.11231869 -2.32424634 -1.77958578 -1.36074698 -0.9691056
 -0.61419933 -0.32544828 -0.01468415  0.26331113  0.60647753  0.91021914
  1.26601326  1.67738299  2.2101442   2.95894032         inf]


### Map each of Alice's values to an interval

In [24]:
def interval_indices(values, thresholds):
    """Return the index of the interval where each real value is"""
    
    return np.digitize(values, thresholds[1:-1], right=False)

In [25]:
A_intervals = [int(x) for x in interval_indices(A_vals, thresholds)]

B_intervals = [int(x) for x in interval_indices(B_vals, thresholds)]

print(f"Sample Alice values: {list(A_vals[:3])}")
print(f"Assigned intervals: {list(A_intervals[:3])}\n")

print(f"Sample Bob values: {list(B_vals[:3])}")
print(f"Assigned intervals: {list(B_intervals[:3])}")

Sample Alice values: [0.6094341595088627, -2.079968212480991, 1.5009023916129145]
Assigned intervals: [10, 2, 12]

Sample Bob values: [-2.315105067702726, -2.1343121604505857, 0.5745518907282836]
Assigned intervals: [2, 2, 9]


In [26]:
def interval_bitstrings(interval_idxs, m):
    
    bit_map = {idx: None for idx in interval_idxs}
    
    bitstr = ""
    
    for idx in interval_idxs:
        
        # calculate binary representation of idx padded to m bits
        bin_rep = bin(idx)[2:].zfill(m)
        
        assert len(bin_rep) == m
        
        bit_map[idx] = bin_rep
        bitstr += bin_rep
    
    return bitstr, bit_map

### Assign bitstring labels to intervals

In [27]:
A_bits, alice_bit_map = interval_bitstrings(A_intervals, m)

print(f"Alice bitstring : {A_bits[:20]} ... | total length {len(A_bits)}")

Alice bitstring : 10100010110011010000 ... | total length 8000


In [28]:
B_bits, bob_bit_map = interval_bitstrings(B_intervals, m)

print(f"Bob bitstring : {B_bits[:20]} ... | total length {len(B_bits)}")

Bob bitstring : 00100010100111000001 ... | total length 8000


In [29]:
def calculate_mismatch(A_bits, B_bits):

    return sum([int("0b" + A_bits[i], 2) ^ int("0b" + B_bits[i], 2) for i in range(len(B_bits))])

In [30]:
errors = calculate_mismatch(A_bits, B_bits)

print(f"Number of different bits in Alice and Bob strings : {errors} out of {len(A_bits)}")

Number of different bits in Alice and Bob strings : 2940 out of 8000
